Refer to this conversation and restartit in chat gpt 

https://chatgpt.com/g/g-p-68932e5ca8788191b76a8d19e9ab4eb1-python-programming/c/69b6ea57-1334-8323-b47a-731893b00453

account: kashyap.rajpurohit

Problem - X 

Intelligent Dictionary & Word Search Engine
Medium → Hard

Data Structures: Trie, Hash Map, DFS, Backtracking, Sorting

You are designing a dictionary engine for an enterprise search application. The system stores thousands of words and must support efficient insertion, lookup, prefix-based retrieval, deletion, autocomplete, and searching for dictionary words inside a two-dimensional character grid.

Initially, the engine supports basic dictionary operations. As new product requirements emerge, you must extend the existing Trie without rebuilding it from scratch for every query.

Your implementation should handle repeated insertions, overlapping prefixes, nonexistent words, and large dictionaries.

words = [
    "app",
    "apple",
    "application",
    "apply",
    "apt",
    "bat",
    "batch",
    "bath",
    "ball",
    "cat",
    "car",
    "card",
    "care"
]

2. Functional requirements — progressively increasing difficulty

Stage	Operation	                                Difficulty	        Key concept

1	    Insert, search, startsWith	                Easy	            Trie fundamentals

2	    Count words sharing prefix	                Medium	            Prefix counters

3	    Delete a word                           	Medium	            Pruning and recursion

4	    utocomplete, top K                      	Medium	            DFS, sorting, ranking

5	    Wildcard search (.)                     	Medium	            DFS branching

6	    Longest common prefix	                    Medium	            Trie traversal

7	    Word Search II	                            Hard	            Trie + grid DFS

8	    Top K frequent suggestions	                Hard	            Heap + Trie

9	    Replace words using shortest root	        Medium	            Prefix matching

10	    Memory and scalability optimization	Staff-level	Design trade-offs

These are the operations the final engine should support.

Operation           	Example	                            Expected output
Exact search        	search("apple")                 	True
Prefix existence	    startsWith("ap")	                True
Prefix counting	        countPrefix("app")	                4
Deletion	            delete("apple")	                    True
Search after deletion	search("apple")	                    False
Autocomplete	        autocomplete("app", 3)	            ["app", "application", "apply"]
Wildcard search	        searchPattern("ba.")	            True
Longest common prefix	longestCommonPrefix(["flower", "flow", "flight"])	"fl"

The autocomplete example assumes lexicographical ordering and runs after "apple" has been deleted.

###  1. Create a Trie 

###  2. Implement utilities like Search word, prefix search, prefix count

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False
        self.prefix_count = 0

class Trie:
    def __init__(self):        
        self.root = TrieNode()

    def insert(self, word):
        # check if the word already exists
        if self.search(word):
            return False
        
        node = self.root 
        node.prefix_count += 1

        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()

            node = node.children[ch]
            node.prefix_count += 1           
        
        node.is_end = True 
        return True

    def search(self, word):
        node = self.root 

        for ch in word:
            if ch not in node.children:
                return False
            node = node.children[ch]

        return node.is_end

    def startswith(self, prefix):
        node = self.root 

        for ch in prefix:
            if ch not in node.children:
                return False 
            node = node.children[ch]

        return True

    # improv 
    def get_prefix_count(self, prefix):
        node = self.root 

        for ch in prefix:
            if ch not in node.children:
                return 0
            node = node.children[ch]
        return node.prefix_count

    # Manually counting 
    def countPrefix(self, prefix:str) -> int:
        node = self.root 

        # locate the prefix node 
        for ch in prefix:
            if ch not in node.children:
                return 0 
            node = node.children[ch]

        # Count completed words belwo prefix - DFS algo
        count = 0 
        stack = [node]
        while stack:
            curr = stack.pop()

            if curr.is_end:
                count += 1

            for child in curr.children.values():
                stack.append(child)
        return count

# Exercise 1: Understand is_end

t1 = Trie()
t1.insert("apple")
print (t1.search("app"))
print (t1.startswith("app"))

t1.insert("app")
print (t1.search("app"))

 # Exercise 2 - Count prefixes
trie = Trie()
trie.insert("app")
trie.insert("apple")
trie.insert("apply")
trie.insert("apt")

print(trie.get_prefix_count("app"))  # 3
print (trie.countPrefix("app"))

print(trie.get_prefix_count("ap"))   # 4
print (trie.countPrefix("ap"))

print(trie.get_prefix_count("xyz"))  # 0
print (trie.countPrefix("xyz"))

print(trie.get_prefix_count("apple"))  # 1
print (trie.countPrefix("apple"))

False
True
True
3
3
4
4
0
0
1
1




###  3. Delete Word 

The below implementationcontains the function for "Deleting a word" froma Trie 

Implement delete(word) so that the requested word is removed, unused nodes are pruned, and all other words sharing its prefix remain searchable. Prefix counts must also be updated correctly.

Edge cases you must handle
- Delete a nonexistent word.
- Delete a word that is a prefix of another word.
- Delete a longer word sharing a prefix with shorter words.
- Delete the last remaining word.
- Delete the same word twice.
- Handle an empty string consistently.

Approach — Iterative deletion using a recorded path
Since we're using Python, I prefer an iterative approach that avoids recursion-depth limitations.

The algorithm:
1. Traverse the word and record each (parent, character, child) relationship.
2. If the word does not exist, return False.
3. Unmark its terminal node.
4. Decrement all affected prefix counters.
5. Walk backward, removing nodes that have no remaining words beneath them.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False
        self.prefix_count = 0

class Trie:
    def __init__(self):        
        self.root = TrieNode()

    def insert(self, word):
        # check if the word already exists
        if self.search(word):
            return False
        
        node = self.root 
        node.prefix_count += 1

        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()

            node = node.children[ch]
            node.prefix_count += 1           
        
        node.is_end = True 
        return True

    def search(self, word):
            node = self.root 
    
            for ch in word:
                if ch not in node.children:
                    return False
                node = node.children[ch]
    
            return node.is_end

    # Delete a word 
    def delete_word(self, word):
        node = self.root
        path = [] 

        # Step 1: Find word and record path
        for ch in word:
            if ch not in node.children:
                return False

            parent = node 
            node = node.children[ch]
            path.append((parent, ch, node))

        # Step 2: Check complete word 
        if not node.is_end:
            return False

        # Step 3: Unmark terminal node 
        node.is_end = False

        # Step 4:  Update prefix Counts
        self.root.prefix_count -= 1

        for parent, ch, child in path:
            child.prefix_count -= 1 

        # Step 5: Prune Unused nodes, bottom up.
        for parent, ch, child in reversed(path):
            if child.prefix_count == 0:
                del parent.children[ch]
            else: 
                break
        return True
    
## Usage
print ("Exercise 3 - Deleting a word")
trie = Trie() 
for word in ["app", "apple", "apply", "apt"]:
    trie.insert(word)

print(trie.delete_word("apple"))       # True
print(trie.search("apple"))       # False
print(trie.search("app"))         # True
print(trie.search("apply"))       # True

print(trie.delete_word("apple"))       # False

### 4. Autocomplete using the same trie 

Given a dictionary of unique words and a prefix, return at most k matching words in lexicographical order.

words = ["app", "apple", "application", "apply", "apt", "bat"]

Expectation:
autocomplete("app", 3)

How to recognize the trie pattern in the question?

The question contains two important requirements:
- Find all words sharing a prefix 
- Return only the first k words in a sorted order

We can locate the prefix node in O(P), then perform DFS through its descendants

Rather than collecting every word and sorting them, we can visit child characters in sorted order and stop once we've found k matches.

Add this method to our existing Trie class


In [13]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False
        self.prefix_count = 0

class Trie:
    def __init__(self):        
        self.root = TrieNode()

    def insert(self, word):
        # check if the word already exists
        if self.search(word):
            return False
        
        node = self.root 
        node.prefix_count += 1

        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()

            node = node.children[ch]
            node.prefix_count += 1           
        
        node.is_end = True 
        return True

    def search(self, word):
            node = self.root 
    
            for ch in word:
                if ch not in node.children:
                    return False
                node = node.children[ch]
    
            return node.is_end    

    # Auto complete 
    def autocomplete(self, prefix, k):
        if k <= 0:
            return []

        node = self.root

        # Locate prefix 
        for ch in prefix:
            if ch not in node.children:
                return []
            node = node.children[ch]

        result = []
        stack = [(node, prefix)]

        # Iterative DFS 
        while stack and len(result) < k:
            curr, word = stack.pop() 

            if curr.is_end:
                result.append(word)

            # Reverse the order because stack is LIFO 
            for ch in sorted(curr.children, reverse=True):
                stack.append((curr.children[ch],word + ch))
        return result
## Usage

print ()
print (f"Testing Auto complete")
trie1 = Trie()
words = ["app", "apple", "application", "apply", "apt", "bat"]
for word in words:
    trie1.insert(word)
print (trie1.autocomplete("app", 3))


Testing Auto complete
['app', 'apple', 'application']


For autocomplete 

Complexity:
- Prefix lookup: O(P) average.
- DFS: O(V + C) in the worst case, where V is the number of visited nodes and C is the total cost of constructing the output strings.
- Sorting children: O(d log d) per visited node, where d is its number of children. For lowercase English letters, d ≤ 26, so this is bounded per node.
- Space: O(V + C) worst case for the stack and returned strings.


### 5. Trie + Wild Card Search (DFS / BackTracking)
LeetCode 211 — Design Add and Search Words Data Structure

This is an important Trie pattern because it introduces branching DFS. Until now, our Trie operations followed exactly one path. With wildcard search, we may need to explore multiple paths.

We'll extend the same Trie implementation from Stages 1–4.

Problem Statement 

You are implementing a dictionary service that supports adding words and searching using patterns.

Design a data structure with two operations:
- addWord(word): Add a word to the dictionary.
- searchPattern(pattern): Return True if any stored word matches the pattern.

The pattern may contain lowercase letters and the special character ..

A dot (.) matches exactly one arbitrary lowercase letter.

Example:
trie.insert("bad")
trie.insert("dad")
trie.insert("mad")

trie.searchPattern("pad")  # False
trie.searchPattern("bad")  # True
trie.searchPattern(".ad")  # True
trie.searchPattern("b..")  # True
trie.searchPattern("..")   # False

Assume words contain lower case a-z and patterns contain lower case a-z plus . 

Analysis:
The defaul search method, when the search word is say 'bad', the path is 
root -> b -> a -> d 

At every step we know which child to visit. But what happens when we encounter.?

For the pattern .ad, since we dont know the first character. We need to explore all possibe children

This changes the problem from single path traversal to branching traversal 

In our example, .ad has three possibilities, but b.. has just the one possibility !

This is precisely why DFS is useful.

**Approach 1: Naive Solution**

Store all the words in a list or a set and compare each word against the pattern.

In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False
        self.prefix_count = 0

class Trie:
    def __init__(self):        
        self.root = TrieNode()
        self.words = set()

    def insert(self, word):
        # check if the word already exists
        if self.search(word):
            return False

        self.words.add(word)
        
        node = self.root 
        node.prefix_count += 1

        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()

            node = node.children[ch]
            node.prefix_count += 1           
        
        node.is_end = True 
        return True

    # word search 
    def search(self, word):
            node = self.root 
    
            for ch in word:
                if ch not in node.children:
                    return False
                node = node.children[ch]
    
            return node.is_end    

    # Approach #1: Naive Solution
    def searchPattern_naive(self, pattern):
        
        for word in self.words:
            if len(word) != len(pattern):
                continue

            if all(p == "." or p == w for p, w in zip(pattern, word)):                
                return True
        return False

    # Optimal Solution 
    def searchPattern(self, pattern):
        stack = [(self.root, 0)]

        while stack:
            node, index = stack.pop()

            if index == len(pattern):
                if node.is_end:
                    return True
                continue

            ch = pattern[index]

            if ch == ".":
                for child in node.children.values():
                    stack.append((child, index + 1))

            elif ch in node.children:
                stack.append((node.children[ch], index + 1))

        return False

"""
Complexity
Let N be the number of stored words and L the pattern length.
- Time: O(N × L) worst case
- Auxiliary space: O(1), excluding the stored dictionary
"""

#### Execution 
trie = Trie()

for word in ["bad", "dad", "mad", "bat", "ball"]:
    trie.insert(word)

patterns = [
    "bad",
    "pad",
    ".ad",
    "b..",
    "..d",
    "...",
    "....",
    ".....",
    "b.t",
    "b.l",
    ""
]

for pattern in patterns:
    naive = trie.searchPattern_naive(pattern)
    dfs = trie.searchPattern(pattern)

    print(f"{pattern!r:8} | Naive: {naive} | DFS: {dfs}")

    assert naive == dfs

'bad'    | Naive: True | DFS: True
'pad'    | Naive: False | DFS: False
'.ad'    | Naive: True | DFS: True
'b..'    | Naive: True | DFS: True
'..d'    | Naive: True | DFS: True
'...'    | Naive: True | DFS: True
'....'   | Naive: True | DFS: True
'.....'  | Naive: False | DFS: False
'b.t'    | Naive: True | DFS: True
'b.l'    | Naive: False | DFS: False
''       | Naive: False | DFS: False


In [19]:
from timeit import timeit

pattern = ".ad"
iterations = 10000

naive_time = timeit(
    lambda: trie.searchPattern_naive(pattern),
    number=iterations
)

dfs_time = timeit(
    lambda: trie.searchPattern(pattern),
    number=iterations
)

print(f"Naive:    {naive_time:.6f} seconds")
print(f"Trie DFS: {dfs_time:.6f} seconds")

Naive:    0.002725 seconds
Trie DFS: 0.003242 seconds


### 6 Trie + DFS + Backtracking (Word Search II, LC 212, Hard).

This is one of the most useful Trie patterns to understand because it combines concepts you've already covered: Trie traversal, DFS, backtracking, pruning, and visited-state management.

We'll follow the same interview approach: problem statement → brute force → optimized solution → dry run → complexity → edge cases.

**Problem Statement:**
Given an m X n grid of characters and a list of words, return all words that can be formed in a grid. 

A word is formed by connecting characters in horizontally or vertically adjacent cells. 

Rules:
- Each cell may be used only once within a word 
- You may move up, down, left, or right 
- Diagonal moves are not allowed
- Return each matching word only once
- Words consist of lower case English letters

board = [
    ["o", "a", "a", "n"],
    ["e", "t", "a", "e"],
    ["i", "h", "k", "r"],
    ["i", "f", "l", "v"]
]

words = ["oath", "pea", "eat", "rain"]

2. How to recognize the pattern

As an interviewer, I'd expect you to identify two separate problems:

Problem A: We must search for words in a 2D grid. This suggests DFS + backtracking.

Problem B: We must search for many words in the same grid. Searching for each word separately repeats a lot of work.

This is where the Trie helps.

Instead of searching the grid separately for every word, we build a Trie containing all words and traverse the grid while simultaneously following valid Trie paths.

The Trie tells us when to stop exploring a path.

For example, if our dictionary contains "oath" and "eat", and a grid traversal produces "ox", there is no reason to continue if "ox" is not a prefix of any dictionary word.

# Naive Approach 
board = [
    ["o", "a", "a", "n"],
    ["e", "t", "a", "e"],
    ["i", "h", "k", "r"],
    ["i", "f", "l", "v"]
]

words = ["oath", "pea", "eat", "rain"]

for word in words:
    for each cell in board:
        run DFS to see if the word exists

This works but repeats grid searches for every word.

If:
- W = number of words
- M × N = board dimensions
- L = maximum word length
A conservative upper bound is: O(W . M . N . 4^L)

We can improve by searching for all words simultaneously.


# Optimal Approach 

1. Insert all the dictionary words in a Trie 
2. Start DFS from every grid cell. 
3. Follow a Trie child only if the current board character exists in node.children
4. If a TRie node marks the end of a word, record the word
5. Mark the current grid cell as vissited 
6. Explore its four neighbours 
7. Restore the cell when returning from DFS 

Why restore the cell?
Suppose one DFS path uses a particular cell. Another independent DFS path may also need that cell.
Marking a cell as visited must apply only to the current path, not to every future search.
That's the essence of backtracking.


In [23]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end = False
        self.prefix_count = 0
        self.word = None

class Trie:
    def __init__(self):        
        self.root = TrieNode()

    def insert(self, word):
        # check if the word already exists
        if self.search(word):
            return False

        node = self.root 
        node.prefix_count += 1

        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()

            node = node.children[ch]
            node.prefix_count += 1           
        
        node.is_end = True 
        return True

    # word search 
    def search(self, word):
            node = self.root 
    
            for ch in word:
                if ch not in node.children:
                    return False
                node = node.children[ch]
    
            return node.is_end

    def findWords(self, board, words):
        if not board or not board[0] or not words:
            return []

        root = TrieNode() 

        # Step1: Build Trie 
        for word in words:
            node = root 

            for ch in word:
                if ch not in node.children:
                    node.children[ch] = TrieNode()

                node = node.children[ch]

            node.word = word
        rows, cols = len(board), len(board[0])
        result = []

        def dfs(r, c, parent):
            ch = board[r][c]

            # Invalid path or already visited
            if ch == "#" or ch not in parent.children:
                return

            node = parent.children[ch]

            # Found a complete word
            if node.word is not None:
                result.append(node.word)
                node.word = None  # Avoid duplicates

            # Mark visited
            board[r][c] = "#"

            # Explore 4 neighbors
            for dr, dc in [(1, 0), (-1, 0),
                           (0, 1), (0, -1)]:
                nr, nc = r + dr, c + dc

                if 0 <= nr < rows and 0 <= nc < cols:
                    dfs(nr, nc, node)

            # Backtrack
            board[r][c] = ch

            # Optional pruning:
            # Remove Trie nodes with no remaining words
            if not node.children and node.word is None:
                del parent.children[ch]

        # Step 2: Start DFS from every cell
        for r in range(rows):
            for c in range(cols):
                if board[r][c] in root.children:
                    dfs(r, c, root)

        return result


### Usage

trie = Trie() 

board = [
    ["o", "a", "a", "n"],
    ["e", "t", "a", "e"],
    ["i", "h", "k", "r"],
    ["i", "f", "l", "v"]
]

words = ["oath", "pea", "eat", "rain"]

print(trie.findWords(board, words))
# ['oath', 'eat'] (order may vary)


['oath', 'eat']


Important: This implementation uses a dedicated Trie for each findWords() call. It doesn't modify the reusable dictionary Trie from our earlier stages.

That is intentional: marking node.word = None and pruning nodes would otherwise change the shared dictionary.

Let S be the total number of characters across dictionary words, and L the maximum word length.
Component	Time
Build Trie	O(S) average
Search grid	O(MN · 4^L) conservative upper bound
Total	O(S + MN · 4^L)

For paths without cell reuse, a tighter conventional bound is approximately O(S + MN · 3^(L−1)) up to constant factors, because after the first move we cannot immediately return to the previous cell.

The actual search is often substantially faster because the Trie prunes prefixes that don't exist in the dictionary.

Space complexity:
- Trie storage: O(S)
- DFS recursion: O(min(L, MN))
- Results: up to O(W) word references, where W is the number of distinct dictionary words.